# Trinity TEFB MC Task - kaggle_benchmarks v2026

Trinity Executive Function Battery - Multiple Choice
Tests multi-step reasoning, stroop interference, wisconsin card sorting, working memory

In [ ]:
# === CELL 1: Install & Fix ===
!pip install protobuf==5.29.6 --quiet
!pip install -q kaggle-benchmarks kaggle

In [ ]:
# === CELL 2: Imports & Config ===
import os
os.environ["RENDER_SUBRUNS"] = "False"

import kaggle_benchmarks as kbench
import kaggle
import pandas as pd
from dataclasses import dataclass
import glob

print("✅ Imports successful")

In [ ]:
# === CELL 3: Download Dataset ===
!mkdir -p /kaggle/working/datasets

kaggle.api.dataset_download_files(
    'playra/trinity-cognitive-probes-tefb',
    path='/kaggle/working/datasets/',
    unzip=True
)

In [ ]:
# === CELL 4: Load MC Data ===
csv_files = glob.glob('/kaggle/working/datasets/**/*.csv', recursive=True)
csv_path = [f for f in csv_files if 'tefb_mc.csv' in f][0]

df = pd.read_csv(csv_path)
# Only use MC questions (skip factual)
df = df[df['question_type'] == 'mc'].copy()
eval_df = df.rename(columns={"question": "question", "choices": "choices", "answer": "expected_answer"})

print(f"📊 Loaded {len(eval_df)} TEFB MC questions")

In [ ]:
# === CELL 5: Structured Output Schema ===
@dataclass
class MCAnswer:
    answer: str

print("✅ Schema defined")

In [ ]:
# === CELL 6: Inner Task (per-item) ===
@kbench.task(name="TEFB Single MC", store_task=False)
def tefb_single(llm, question: str, choices: str, expected_answer: str) -> bool:
    prompt = f"""{question}

{choices}

Answer with ONLY ONE letter (A, B, C, or D)."""
    resp = llm.prompt(prompt, schema=MCAnswer)
    got = resp.answer.strip().upper()[0]
    exp = str(expected_answer).strip().upper()[0]
    return got == exp

print("✅ Inner task registered")

In [ ]:
# === CELL 7: Outer Benchmark Task ===
@kbench.task(name="Trinity TEFB MC", description="Trinity Executive Function Battery - Multiple Choice")
def tefb_benchmark(llm) -> float:
    with kbench.client.enable_cache():
        runs = tefb_single.evaluate(
            llm=[llm],
            evaluation_data=eval_df,
            n_jobs=2,
            timeout=180,
            max_attempts=1,
            remove_run_files=True,
        )
    df_res = runs.as_dataframe()
    valid = df_res[df_res["result"].notna()]
    acc = float(valid["result"].mean())

    kbench.assertions.assert_true(
        True,
        expectation=f"TEFB accuracy: {acc:.2%} ({len(valid)}/{len(eval_df)})"
    )
    return acc

print("✅ Outer benchmark task registered")

In [ ]:
# === CELL 8: Run & Choose ===
run = tefb_benchmark.run()
print(f"\n🏆 Result: {run.result:.2%}")
%choose tefb_benchmark